선형 회귀 모델LinearRegrssion

사이킷런 선형모델 선형회귀모델

model <=

fit(X,Y) -> 회귀선

model predict -> 특정 X에 대한 Y값을 예측

회귀모델은 추세trend와 계절성을 사람이 변수로 만들어서 입력해야 한다.

시계열 전용 모델이 있는데 시계열 전용 모델은 최근 값과 최근 오차가 다음 값에 어떻게 이어지는가를 데이터에서 스스로 찾는다(학습한다).

- SARIMA
- 지수평활exponential smoothing




## SARIMA
ARIMA, AR(자기회귀autoregressive), I(누적integrated), MA(이동평균 오차moving average)

자기회귀는 지난 달 값이 높았으면 이번 달도 높다처럼 과거 값으로 현재를 설명하는 것이고 누적은 값 대신 전월과의 차이(차분)를 모델링해서 추세를 없애고 이동평균은 지난 달 예측이 빗나간만큼 이번달 값을 보정하겠다.

# 모델의 평가 설계, 개선
모델이 예측한 결과를 얼마나 믿을 수 있는지를 따져볼 수 있는 도구가 필요하다. 이때 RMSE, ME, MASE 같은 평가 지표를 더하고 예측기간별로 오차를 나누어보는 롤링 원점 평가를 설계하고 로그변화 회귀, 계절시차 특성, 학습기간선택, 예측/결합 등의 방법으로 모델을 다듬을 수 있다.

## 지표
- RMSE root mean squared error평균제곱근 오차: 큰 오차가 얼마나 있는가?
- ME mean error편향: 한쪽으로 치우쳐져서 예측하는가?
- MASE mean absolute scaled error: 계절 나이브보다 얼마나 나은가?

### RMSE
RMSE는 오차를 제곱해서 평균을 구한 다음에 다시 제곱근을 씌운 값을 의미한다. 제곱을 하게되면 큰 오차가 더 커지게 된다. 그래서 RMSE를 구하게 되면 큰 오차에 민감하게 반응할 수 있게 된다.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pygments.styles import native
from sklearn.linear_model import LinearRegression
plt.rcParams['font.family'] = 'Malgun Gothic'

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr')

In [4]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
# RMSE, MAE: 늘 조금씩 틀리는 예측, 가끔 크게 틀리는 예측

actual = np.full(12, 2_000_000)
steady = actual+ np.array([50_000, -50_000]*6)
spiky = actual.copy()
spiky[6] = actual[6] + 550_000
spiky = spiky +np.array([1_000]*6 + [0] + [1_000]*5)

errors = actual - spiky
print(f'MAE = {round(np.abs(errors).mean())}') #평균의 오류
#RMSE
#sqrt()
print(f'RMSE = {round(np.sqrt((errors ** 2).mean()))}')#잘못된 게 확 드러남, 큰 거에 제곱하면 뻥뛰기됨


MAE = 46750
RMSE = 158774


In [6]:
for name, pred in [('늘 조금씩 틀림', steady), ('가끔 크게 틀림', spiky)]:
    mae = mean_absolute_error(actual, pred) # 모델이 예측한 값
    rmse = root_mean_squared_error(actual, pred)
    print(f'{name} 모델의 MAE = {mae:,.0f} RMSE = {rmse:,.0f}')

늘 조금씩 틀림 모델의 MAE = 50,000 RMSE = 50,000
가끔 크게 틀림 모델의 MAE = 46,750 RMSE = 158,774


## 편향 bias
예측에서도 오차가 위아래로 고르게 흩어지는 것과 한쪽으로 몰릴 때 조치해야 되는 내용이 다르다. 한쪽으로 치우친 정도를 편향bias이라고 일컫는다. 오차에 절댓값을 씌우지 않고 그대로 평균을 구하게 되면 평균 오차ME mean error 라고 일컫는다.

편향이 양수이면 모델이 실제보다 작게 예측하는 경향이 있고 음수면 크게 예측하는 경향이 있다. MAE 가운데 편향이 차지하는 몫이 크면 모델이 추세나 수준을 잘못 잡고 있다고 판단할 수 있고 이러면 모델 구조를 고쳐야 된다. 편향이 0에 가깝고 MAE만 크면 설명하지 못하는 불규칙 변동이 크다고 볼 수 있다.

10만 , -10만의 오차가 합산되면 0이 된다. 편향이 없는 것처럼 보인다. 모델의 예측은 정확하지 않는 것이다. 편향은 편향만 구해서 사용하는 것이 아니라 MAE와 같이 사용한다.

- 편향 = 평균(실제값-예측값)
    - 양수: 과소 예측
    - 음수: 과대 예측

## MASE mean absolute scaled error
MAE가 5만 TEU라는 결과가 나왔을 때 해댱 지표가 좋은 지표인지 나쁜 지표인지를 판단하기 어렵다. 월 물동량 200만 TEU에서는 오차가 2.5% 정도이기 때문에 좋은 상태라고 판단해 볼 수 있지만 다른 수단으로 4만 TEU를 쉽게 구할 수 있었다면 이 지표는 좋지 않은 지표로 볼 수 있는 것이다.

MASE는 이러한 비교를 한 숫자로 만든 지표로서 모델의 MAE를 게절 나이브 학습 구간에서 보인 평균 오차로 나눈 값.

In [11]:
season = [-40, -120, 60, 40, 50, 10, 30, -10, -60,10, -10, -10]
# 48개월 변화량을 보여주는 것-패턴 매년 이런 움직임을 가짐
noise = np.random.default_rng(42).normal(0,15,48).round(0) #이걸 위에 넣음 42=난수 생성기 /난수 생성 알고리즘에 의해 씨드 결정
idx = pd.date_range('2021-01-01',periods=48,freq='MS')

toy = pd.Series([ 1900+5 * i + season[i % 12] for i in range(48)], index=idx)
#1%12 모듈러 연산자 -> 12보다 클 수가 없음 0부터 11까지 게속 반복적으로 돌게 하기 위해
#계절 패턴 데이터

toy

2021-01-01    1860
2021-02-01    1785
2021-03-01    1970
2021-04-01    1955
2021-05-01    1970
2021-06-01    1935
2021-07-01    1960
2021-08-01    1925
2021-09-01    1880
2021-10-01    1955
2021-11-01    1940
2021-12-01    1945
2022-01-01    1920
2022-02-01    1845
2022-03-01    2030
2022-04-01    2015
2022-05-01    2030
2022-06-01    1995
2022-07-01    2020
2022-08-01    1985
2022-09-01    1940
2022-10-01    2015
2022-11-01    2000
2022-12-01    2005
2023-01-01    1980
2023-02-01    1905
2023-03-01    2090
2023-04-01    2075
2023-05-01    2090
2023-06-01    2055
2023-07-01    2080
2023-08-01    2045
2023-09-01    2000
2023-10-01    2075
2023-11-01    2060
2023-12-01    2065
2024-01-01    2040
2024-02-01    1965
2024-03-01    2150
2024-04-01    2135
2024-05-01    2150
2024-06-01    2115
2024-07-01    2140
2024-08-01    2105
2024-09-01    2060
2024-10-01    2135
2024-11-01    2120
2024-12-01    2125
Freq: MS, dtype: int64

In [13]:
train, test = toy.iloc[:36], toy.iloc[36:]
len(train)
len(test)

12

In [14]:
# 분모: 학습 구간에서 계절 나이브(12개월전 값)의 평균 절대 오차
scale = (train-train.shift(12)).abs().mean()
print(round(scale,1)) #매년 60정도의 오차가 난다

60.0


In [15]:
def mase(actual, pred, scale):
    return mean_absolute_error(actual, pred) /scale


In [16]:
seasonal_naive = train.iloc[-12:].to_numpy() #작년 값은 달 값
with_growth = seasonal_naive + 12 * 5 # 작년 같은 달 +1년치 추세(월 5천 TEU * 12)

In [19]:
mase(test, seasonal_naive, scale) #이게 더 좋은 데이터구나..)
mase(test, with_growth, scale)

np.float64(0.0)

## 예측기간horizon
에측기간은 예측을 만든 시점에서 몇 칸 앞의 값을 예측하는가를 의미한다.

2월 말에 3월을 예측하면 1개월 앞에 있는 예측(h=1), 다음해 2월의 값을 예측 12개월(h=12)이 된다.

일기예보를 두고 볼 때 내일 것은 잘 맞는 편이나 열흘 뒤의 예보를 보면 상대적으로 잘 안 맞는다. 예측은 먼 미래일수록 그 사이에 생길 수 있는 일이 많아지기 때문에 오차가 커질 수 밖에 없다.

예측 오차는 예측 기간별로 따로따로 살펴 보아야 한다. MAE가 5만이다 보다는 1개월 앞의 MAE가 4만, 12개월 앞의 MAE는 7만이라는 내용이 잘 맞는 표현이다.

In [20]:
origin = train.index[-1] # 예측을 만드는 시점
print(origin)

2023-12-01 00:00:00


In [21]:
horizon = np.arange(1,13)
print(horizon)

[ 1  2  3  4  5  6  7  8  9 10 11 12]


In [27]:
train.iloc[-1]

np.int64(2065)

In [26]:
naive = np.full(12, train.iloc[-1])
print(naive)

[2065 2065 2065 2065 2065 2065 2065 2065 2065 2065 2065 2065]


In [24]:
abs_err = pd.DataFrame({
    'h':horizon,
    'naive': np.abs(test.to_numpy()-native),
    'seasonality naive': np.abs(test.to_numpy()-seasonal_naive)}).set_index('h').round()


abs_err

,naive,seasonality naive
h,,
1,25,60
2,100,60
3,85,60
4,70,60
5,85,60
6,50,60
7,75,60
8,40,60
9,5,60


## 롤링 원점 평가
롤링원점평가는 예측을 시작하는 시점(원점)을 한달씩 뒤로 옮기면서 매번 원점까지의 데이터 학습, 원점 뒤 h개월 예측, 실제값과 비교를 반복.

In [25]:
def seasonal_naive_forecast(history , horizon):
    return history.iloc[-12:].to_numby()[:horizon]

def rolling_origin(s, forecaster, origins, horizon=12):
    records = []

    for origin in origins:
        history = s.loc[:origin]
        future = s.loc[origin:]
        pred = forecaster(history, horizon)[:len(future)]
        for h, (date, actual_value) in enumerate(future.items(),start=1):
            records.append({'원점': origin, 'h': h, '날짜': date, '실제값': actual_value, '예측값': pred[h-1]})
    return pd.DataFrame(records)